# 2D Percolation and Phase Transition Analysis

This notebook explores **site percolation** on a 2-D square lattice and identifies the
percolation threshold $p_c \approx 0.5927$.  
It uses:
- A custom **PCG32** random-number generator (`src/rng_engine.py`) JIT-compiled with **Numba**.
- A **Numba**-accelerated grid generator and **Scipy** cluster labeller (`src/simulation.py`).
- **Matplotlib** for visualisation.

## Scientific context

The percolation model is directly relevant to **stochastic star formation**:  
molecular clouds are represented as lattice sites that are *activated* (occupied) with
probability $p$.  When a giant connected component first spans the lattice the
large-scale star-formation episode can begin — analogous to the infinite cluster
appearing at $p_c$.  The ratio $L_2 / L_1$ peaks sharply near $p_c$ and provides a
robust finite-size estimator of the threshold.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from numba import uint64

from src.simulation import generate_grid, label_clusters, get_l1_l2, run_simulation

plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

## 1  Visualise a single lattice at the critical probability

In [ ]:
L = 100
p_c = 0.5927

grid = generate_grid(L, p_c, uint64(42), uint64(1))
labeled, n_clusters = label_clusters(grid)
L1, L2 = get_l1_l2(labeled, n_clusters)

print(f'Lattice size      : {L}×{L}')
print(f'Occupation prob.  : {p_c}')
print(f'Number of clusters: {n_clusters}')
print(f'L1 (largest)      : {L1}')
print(f'L2 (2nd largest)  : {L2}')
print(f'L2/L1             : {L2/L1:.4f}' if L1 > 0 else 'L2/L1: N/A')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(grid, cmap='binary', interpolation='nearest')
axes[0].set_title(f'Occupancy grid  (p = {p_c})')
axes[0].axis('off')

axes[1].imshow(labeled, cmap='nipy_spectral', interpolation='nearest')
axes[1].set_title(f'Labelled clusters  (n = {n_clusters})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2  Scan over probability: L1, L2 and L2/L1 vs p

In [ ]:
L = 100
p_values = np.linspace(0.30, 0.80, 51)
n_samples = 10  # realisations per p value

l1_mean = np.zeros_like(p_values)
l2_mean = np.zeros_like(p_values)
ratio_mean = np.zeros_like(p_values)

for idx, p in enumerate(p_values):
    l1_list, l2_list, ratio_list = [], [], []
    for s in range(n_samples):
        L1, L2, _, _ = run_simulation(L, p, seed=idx * 100 + s, inc=1)
        l1_list.append(L1)
        l2_list.append(L2)
        ratio_list.append(L2 / L1 if L1 > 0 else 0.0)
    l1_mean[idx] = np.mean(l1_list)
    l2_mean[idx] = np.mean(l2_list)
    ratio_mean[idx] = np.mean(ratio_list)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(p_values, l1_mean / L**2, 'b-o', ms=4)
axes[0].axvline(0.5927, color='red', ls='--', label='$p_c$')
axes[0].set_xlabel('p'); axes[0].set_ylabel('$L_1 / L^2$')
axes[0].set_title('Largest cluster fraction'); axes[0].legend()

axes[1].plot(p_values, l2_mean / L**2, 'g-s', ms=4)
axes[1].axvline(0.5927, color='red', ls='--', label='$p_c$')
axes[1].set_xlabel('p'); axes[1].set_ylabel('$L_2 / L^2$')
axes[1].set_title('Second-largest cluster fraction'); axes[1].legend()

axes[2].plot(p_values, ratio_mean, 'r-^', ms=4)
axes[2].axvline(0.5927, color='red', ls='--', label='$p_c$')
axes[2].set_xlabel('p'); axes[2].set_ylabel('$L_2 / L_1$')
axes[2].set_title('Ratio $L_2/L_1$ (peaks at $p_c$)'); axes[2].legend()

plt.tight_layout()
plt.show()

## 3  Finite-size scaling: locate p_c from L2/L1 peak for several L

In [ ]:
lattice_sizes = [20, 40, 80]
p_values_fine = np.linspace(0.50, 0.68, 37)
n_samples = 20

fig, ax = plt.subplots(figsize=(8, 5))

for L in lattice_sizes:
    ratio_mean = np.zeros(len(p_values_fine))
    for idx, p in enumerate(p_values_fine):
        ratios = []
        for s in range(n_samples):
            L1, L2, _, _ = run_simulation(L, p, seed=idx * 1000 + s * 7, inc=1)
            ratios.append(L2 / L1 if L1 > 0 else 0.0)
        ratio_mean[idx] = np.mean(ratios)

    peak_idx = np.argmax(ratio_mean)
    p_peak = p_values_fine[peak_idx]
    ax.plot(p_values_fine, ratio_mean, '-o', ms=4, label=f'L={L}  ($\\hat{{p}}_c$={p_peak:.4f})')

ax.axvline(0.5927, color='k', ls='--', lw=1.5, label='Exact $p_c = 0.5927$')
ax.set_xlabel('p')
ax.set_ylabel('$L_2 / L_1$')
ax.set_title('Finite-size scaling of $L_2/L_1$')
ax.legend()
plt.tight_layout()
plt.show()